# Part 2, Topic 1, Lab A: Instruction Power Differences – HARDWARE Setup

---
**This notebook is `%run` automatically by the `(MAIN)` notebook when `VERSION = 'HARDWARE'`.**
You normally don't open it directly - set `VERSION` and the target options in `(MAIN)` instead.

---

What it does:

1. Makes a private copy of the `simpleserial-base` firmware at
   `firmware/mcu/simpleserial-base-lab2` so edits here don't touch the original.
2. Connects to the ChipWhisperer and runs `scope.default_setup()`.
3. Builds and flashes a baseline build so `capture_trace()` works right away.
4. Provides `set_user_code(...)` + `build_and_flash()` for the MAIN notebook to
   change the instructions under test between captures, and `capture_trace()`.

Defaults (`PLATFORM='CWHUSKY'`) match the ChipWhisperer-Husky kit (SAM4S on CW313).
For a CW308 UFO board use `PLATFORM='CW312_SAM4S'`; for STM32F3 use `'CW308_STM32F3'`.

## Firmware working copy

In [ ]:
import os, shutil, subprocess, pathlib

FW_SRC = pathlib.Path('../../../firmware/mcu/simpleserial-base')
FW_DIR = pathlib.Path('../../../firmware/mcu/simpleserial-base-lab2')
FW_C   = FW_DIR / 'simpleserial-base.c'

if not FW_DIR.exists():
    shutil.copytree(FW_SRC, FW_DIR)
_FW_C_PRISTINE = (FW_SRC / 'simpleserial-base.c').read_text()
FW_C.write_text(_FW_C_PRISTINE)   # start every run from a clean copy
print('firmware working copy:', FW_DIR.resolve())

## Connect to the ChipWhisperer

In [ ]:
import chipwhisperer as cw
import time

try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

_ttype = cw.targets.SimpleSerial2 if SS_VER == 'SS_VER_2_1' else cw.targets.SimpleSerial
try:
    target = cw.target(scope, _ttype)
except IOError:
    print('INFO: reconnecting to scope first (USB hiccup) - ignore errors above this line')
    scope = cw.scope()
    target = cw.target(scope, _ttype)

print('INFO: Found ChipWhisperer😍')

if 'STM' in PLATFORM or PLATFORM in ('CWLITEARM', 'CWNANO'):
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM in ('CW303', 'CWLITEXMEGA'):
    prog = cw.programmers.XMEGAProgrammer
elif 'neorv32' in PLATFORM.lower():
    prog = cw.programmers.NEORV32Programmer
elif PLATFORM in ('CW308_SAM4S', 'CW312_SAM4S', 'CWHUSKY'):
    prog = cw.programmers.SAM4SProgrammer
else:
    prog = None

time.sleep(0.05)
scope.default_setup()

if PLATFORM in ('CW308_SAM4S', 'CW312_SAM4S', 'CWHUSKY'):
    scope.io.target_pwr = 0; time.sleep(0.2)
    scope.io.target_pwr = 1; time.sleep(0.3)

def reset_target(scope):
    if PLATFORM in ('CW303', 'CWLITEXMEGA'):
        scope.io.pdic = 'low';  time.sleep(0.1)
        scope.io.pdic = 'high_z'; time.sleep(0.1)
    else:
        scope.io.nrst = 'low';  time.sleep(0.25)
        scope.io.nrst = 'high_z'; time.sleep(0.25)

## Firmware-edit helpers

`get_pt()` in `simpleserial-base.c` has a block between `trigger_high()` and
`trigger_low()` marked *"Start user-specific code here"*. `set_user_code()` drops C
statements into that block; `build_and_flash()` recompiles and programs the target.

In [ ]:
_TRIG_HI = 'trigger_high();'
_TRIG_LO = 'trigger_low();'

def set_user_code(c_statements, keep_serial_echo=True):
    """Insert C statements into the get_pt() trigger window of the working copy."""
    src = _FW_C_PRISTINE
    hi = src.index(_TRIG_HI) + len(_TRIG_HI)
    lo = src.index(_TRIG_LO)
    src = src[:hi] + ' ' + c_statements + ' ' + src[lo:]
    if not keep_serial_echo:
        src = src.replace("simpleserial_put('r', 16, pt);", "// serial echo removed")
    FW_C.write_text(src)

def build_and_flash():
    r = subprocess.run(['make', 'PLATFORM=' + PLATFORM, 'CRYPTO_TARGET=NONE',
                        'SS_VER=' + SS_VER, '-j'],
                       cwd=FW_DIR, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError('firmware build failed')
    cw.program_target(scope, prog, str(FW_DIR / ('simpleserial-base-{}.hex'.format(PLATFORM))))

## Live capture + baseline flash

In [ ]:
def capture_trace(_section=None):
    """Capture one power trace from the target (the _section arg is ignored on hardware)."""
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    for _ in range(3):
        tr = cw.capture_trace(scope, target, text)
        if tr is not None:
            return tr.wave
    raise IOError('capture_trace timed out 3x - check trigger (CW313 JP2 = T-GPIO4) and target')

# Flash a clean baseline so the MAIN notebook's first capture works before it
# calls build_and_flash() for section 1.
set_user_code('', keep_serial_echo=True)
build_and_flash()
print('✔️ Hardware ready - baseline simpleserial-base flashed.')

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2020. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>